# Global Factor Premiums

## Citation

**Title:** Global factor premiums  
**Authors:** Guido Baltussen, Laurens Swinkels, Pim van Vliet  
**Published:** 2021-07-01  
**URL:** [https://doi.org/10.1016/j.jfineco.2021.06.030](https://doi.org/10.1016/j.jfineco.2021.06.030)

**Abstract:**
We examine 24 global factor premiums across equity, bond, commodity, and currency markets via replication and out-of-sample evidence between 1800 and 2016. Replication yields ambiguous evidence within a unified testing framework that accounts for p-hacking. Out-of-sample tests reveal strong and robust presence of the large majority of global factor premiums, with limited out-of-sample decay of the premiums. We find global factor premiums to be generally unrelated to market, downside, or macroeconomic risks in the 217 years of data. These results reveal significant global factor premiums that present a challenge to traditional asset pricing theories.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration parameters
ticker = 'SPY'
start_date = '2010-01-01'
end_date = '2020-12-31'

## Phase 2: Data Download and Feature Engineering

In [ ]:
# Download historical data
data = yf.download(ticker, start=start_date, end=end_date)

# Feature engineering: Calculate daily returns
data['Return'] = data['Adj Close'].pct_change()

# Drop NaN values
data.dropna(inplace=True)

## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
# Simple moving average strategy
short_window = 40
long_window = 100

data['SMA40'] = data['Adj Close'].rolling(window=short_window, min_periods=1).mean()
data['SMA100'] = data['Adj Close'].rolling(window=long_window, min_periods=1).mean()

# Generate signals
data['Signal'] = 0
# Buy signal
mask_buy = data['SMA40'] > data['SMA100']
data.loc[mask_buy, 'Signal'] = 1
# Sell signal
mask_sell = data['SMA40'] < data['SMA100']
data.loc[mask_sell, 'Signal'] = -1

## Phase 4: Vectorized Backtest

In [ ]:
# Shift signals to avoid look-ahead bias
data['Position'] = data['Signal'].shift(1)

# Calculate strategy returns
data['Strategy_Return'] = data['Position'] * data['Return']

# Calculate cumulative returns
data['Cumulative_Strategy_Return'] = (1 + data['Strategy_Return']).cumprod()
data['Cumulative_Market_Return'] = (1 + data['Return']).cumprod()

## Phase 5: Performance Metrics

In [ ]:
# Calculate performance metrics
def calculate_performance_metrics(data):
    # Sharpe Ratio
    sharpe_ratio = data['Strategy_Return'].mean() / data['Strategy_Return'].std() * np.sqrt(252)
    
    # Sortino Ratio
    downside_std = data[data['Strategy_Return'] < 0]['Strategy_Return'].std()
    sortino_ratio = data['Strategy_Return'].mean() / downside_std * np.sqrt(252)
    
    # Max Drawdown
    roll_max = data['Cumulative_Strategy_Return'].cummax()
    daily_drawdown = data['Cumulative_Strategy_Return'] / roll_max - 1.0
    max_drawdown = daily_drawdown.cummin().min()
    
    # Calmar Ratio
    calmar_ratio = data['Strategy_Return'].mean() * 252 / abs(max_drawdown)
    
    return sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown = calculate_performance_metrics(data)

# Print performance metrics
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")

## Phase 6: Monitoring Stub

In [ ]:
# Placeholder for monitoring logic
# This could include real-time data fetching, alerting, etc.

print("Monitoring phase is under development.")